# Pipeline de Análise Empírica — Benchmarking e Telemetria de Hardware

**Disciplina:** Arquitetura e Organização de Computadores
**Curso:** Engenharia de Computação — UFPA (Campus Universitário de Tucuruí)
**Professor orientador:** Prof. Dr. Iago Medeiros

Este notebook orquestra o pipeline completo de análise dos dados experimentais
coletados pelo grupo em **4 (ou mais) máquinas** distintas, cada uma submetida a
**20 rodadas** do *Geekbench 6* com telemetria simultânea via **HWiNFO64**.

O notebook é dividido nas seguintes etapas:

1. Configuração do ambiente e localização da estrutura de pastas do projeto.
2. **Verificação de completude dos dados** (cada máquina deveria ter, no
   máximo, 1 arquivo `.txt` de scores + 20 arquivos `.csv` de telemetria).
3. Análise estatística dos **resultados de benchmark** (Single-Core / Multi-Core).
4. Análise estatística da **telemetria de hardware** (clock, temperatura,
   potência, throttling, memória).
5. **Cruzamento arquitetural**: Desempenho por Watt e IPC relativo.
6. Geração dos **gráficos no padrão visual da SBC** (escala de cinza, sóbrio).
7. Exportação das tabelas consolidadas para uso direto no artigo (LaTeX/SBC).

> **Importante:** o pipeline foi projetado para funcionar mesmo com dados
> **parciais** (máquinas com menos de 20 rodadas, ou ainda sem nenhum dado).
> Sempre que uma máquina/rodada estiver incompleta, um aviso é exibido no
> log, mas a execução **não é interrompida** — a análise é refeita
> automaticamente conforme novos arquivos forem adicionados às pastas em
> `data/raw/machine_x`.


## Etapa 0 — Configuração do ambiente

Adicionamos a pasta `src/` ao `sys.path` para podermos importar os 4 módulos
do pipeline (`utils_utils`, `benchmark_analysis`, `telemetry_analysis`,
`plotting_engine`) como se fossem bibliotecas comuns.


In [1]:
# Recarregamento automático ao editar os .py
%load_ext autoreload
%autoreload 2

In [2]:
import importlib.util
import subprocess
import sys
from pathlib import Path

# =====================================================================
# 1. RESOLUÇÃO DE CAMINHOS DO PROJETO (MANUTENÇÃO DO SEU FLUXO)
# =====================================================================
RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / "src").is_dir() and (RAIZ_PROJETO.parent / "src").is_dir():
    RAIZ_PROJETO = RAIZ_PROJETO.parent

CAMINHO_SRC = RAIZ_PROJETO / "src"
if str(CAMINHO_SRC) not in sys.path:
    sys.path.append(str(CAMINHO_SRC))

print("Raiz do projeto detectada em:", RAIZ_PROJETO)
print("Pasta 'src' adicionada ao sys.path:", CAMINHO_SRC)

Raiz do projeto detectada em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark
Pasta 'src' adicionada ao sys.path: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\src


In [3]:
# =====================================================================
# 2. VERIFICAÇÃO E INSTALAÇÃO DE DEPENDÊNCIAS (SOLUÇÃO JINJA2 + GRÁFICOS)
# =====================================================================
PACOTES_REQUISITOS = ["pandas", "numpy", "jinja2", "matplotlib", "seaborn"]

print("\nVerificando dependências do pipeline...")
for pacote in PACOTES_REQUISITOS:
    if importlib.util.find_spec(pacote) is None:
        print(f" 📦 Pacote '{pacote}' não encontrado. Instalando agora...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pacote, "--quiet"])
    else:
        print(f" ✅ Pacote '{pacote}' já está instalado.")


Verificando dependências do pipeline...
 ✅ Pacote 'pandas' já está instalado.
 ✅ Pacote 'numpy' já está instalado.
 ✅ Pacote 'jinja2' já está instalado.
 ✅ Pacote 'matplotlib' já está instalado.
 ✅ Pacote 'seaborn' já está instalado.


In [4]:
# =====================================================================
# 3. IMPORTAÇÃO DOS MÓDULOS E CONFIGURAÇÃO DO AMBIENTE
# =====================================================================
import pandas as pd
import numpy as np

# Importações dos módulos internos (agora garantidos pelo sys.path acima)
import utils_utils as u
import benchmark_analysis as bench
import telemetry_analysis as tele
import plotting_engine as plot

# Configurações de exibição do Pandas
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

# Inicialização do estilo científico de gráficos
plot.configurar_estilo_sbc()
print("\n🚀 Todos os módulos e dependências foram carregados com sucesso.")


🚀 Todos os módulos e dependências foram carregados com sucesso.


## Etapa 1 — Verificação da estrutura e completude dos dados

Antes de qualquer análise, **validamos a coleta**: cada pasta `data/raw/machine_x`
deve conter, no máximo, **20 arquivos `.csv`/`.CSV`** (1 por rodada de telemetria) e
**1 arquivo `.txt`** (scores consolidados do Geekbench 6) — **21 arquivos no total**.

O pipeline aceita **menos** arquivos (coleta ainda em andamento) mas emite um
**alerta** caso encontre **mais** do que o esperado (sinal de erro na coleta,
como rodadas duplicadas ou arquivos de outra máquina na pasta errada).

> Edite `NOMES_EXIBICAO` abaixo conforme as máquinas forem identificadas
> (ex.: notebook da Roberta, desktop do laboratório etc.).


In [5]:
# Mapeamento opcional: nome da pasta -> nome de exibição (use nos gráficos/tabelas)
NOMES_EXIBICAO = {
    "machine_a": "Máquina A",
    "machine_b": "Máquina B",
    "machine_c": "Máquina C",
    "machine_d": "Máquina D",
    "machine_e": "Máquina E",
    "machine_f": "Máquina F",
}

caminhos = u.obter_caminhos(str(RAIZ_PROJETO))
relatorios = u.verificar_estrutura_diretorios(caminhos)

if not relatorios:
    print(
        "Nenhuma pasta 'machine_*' encontrada em data/raw/. "
        "Verifique se os dados já foram adicionados ao repositório."
    )
else:
    df_completude = u.exibir_relatorio_completude(relatorios)
    display(df_completude)


[01:18:42] INFO     | aoc_pipeline | machine_a    | TXT: 1/1 | CSV: 20/20 | Completude: 100.0% | OK (completa)
[01:18:42] INFO     | aoc_pipeline | machine_b    | TXT: 1/1 | CSV: 20/20 | Completude: 100.0% | OK (completa)
[01:18:42] INFO     | aoc_pipeline | machine_c    | TXT: 1/1 | CSV: 20/20 | Completude: 100.0% | OK (completa)
[01:18:42] INFO     | aoc_pipeline | machine_d    | TXT: 1/1 | CSV: 20/20 | Completude: 100.0% | OK (completa)
[01:18:42] INFO     | aoc_pipeline | machine_e    | TXT: 1/1 | CSV: 20/20 | Completude: 100.0% | OK (completa)
[01:18:42] INFO     | aoc_pipeline | machine_f    | TXT: 1/1 | CSV:  0/20 | Completude:   4.8% | INCOMPLETA/ATENÇÃO
[01:18:42] WARNING  | aoc_pipeline |   -> [machine_f] Nenhum arquivo .csv de telemetria (HWiNFO64) encontrado nesta máquina.


,maquina,arquivos_txt,arquivos_csv,total_arquivos,percentual_completude,completa,qtd_alertas
0,machine_a,1,20,21,100.0,True,0
1,machine_b,1,20,21,100.0,True,0
2,machine_c,1,20,21,100.0,True,0
3,machine_d,1,20,21,100.0,True,0
4,machine_e,1,20,21,100.0,True,0
5,machine_f,1,0,1,4.8,False,1


In [6]:
# Tabela de completude pronta para exportação (ex.: anexo metodológico do artigo)
if relatorios:
    caminho_saida = caminhos.dados_processed / "relatorio_completude.csv"
    df_completude.to_csv(caminho_saida, index=False, encoding="utf-8")
    print(f"Relatório de completude salvo em: {caminho_saida}")


Relatório de completude salvo em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\data\processed\relatorio_completude.csv


In [50]:
import pandas as pd

caminho_arquivo = "data\\raw\\machine_a\\maqA_rodada_01.CSV"

# CORREÇÃO: UTF-8 limpa os acentos e errors="replace" impede o travamento na linha com erro
df = pd.read_csv(
    caminho_arquivo, 
    encoding="utf-8", 
    on_bad_lines='skip', 
    encoding_errors='replace',
)

# Exibe o cabeçalho corrigido para testar
print([col for col in df.columns if "Utilização" in col])


['Utilização do arquivo de paginação [%]', 'Utilização do núcleo (avg) [%]', 'P-core 0 T0 Utilização [%]', 'P-core 0 T1 Utilização [%]', 'P-core 1 T0 Utilização [%]', 'P-core 1 T1 Utilização [%]', 'P-core 2 T0 Utilização [%]', 'P-core 2 T1 Utilização [%]', 'P-core 3 T0 Utilização [%]', 'P-core 3 T1 Utilização [%]', 'E-core 4 T0 Utilização [%]', 'E-core 5 T0 Utilização [%]', 'E-core 6 T0 Utilização [%]', 'E-core 7 T0 Utilização [%]', 'Utilização total da CPU [%]', 'Utilização de D3D GPU [%]', 'Utilização [Yes/No]']


In [51]:
# 2. Identifica quais colunas possuem 100% de valores nulos/vazios
colunas_vazias = df.columns[df.isna().all()].tolist()

# 3. Exibe o resultado no console
print(f"Total de colunas completamente vazias: {len(colunas_vazias)}")
print("Lista de colunas vazias:", colunas_vazias)


Total de colunas completamente vazias: 1
Lista de colunas vazias: ['Unnamed: 311']


In [52]:
# 2. Identifica as linhas onde TODAS as colunas são NaN
linhas_vazias_mask = df.isna().all(axis=1)

# 3. Extrai os números das linhas (índices) que estão vazias
indices_linhas_vazias = df.index[linhas_vazias_mask].tolist()

print(f"Total de linhas completamente vazias: {len(indices_linhas_vazias)}")
print("Índices das linhas vazias:", indices_linhas_vazias)


Total de linhas completamente vazias: 0
Índices das linhas vazias: []


In [ ]:
# Remove apenas as linhas que estão 100% vazias
#df_limpo = df.dropna(how="all")

# Se quiser salvar o arquivo corrigido por cima do antigo:
# df_limpo.to_csv("caminho/do/seu/arquivo.csv", index=False)


In [55]:
import numpy as np
import pandas as pd

# 1. Filtra apenas colunas numéricas (telemetria quantitativa)
df_numerico = df.select_dtypes(include=[np.number]).copy()

colunas_relevantes = []
colunas_constantes = []
colunas_quase_vazias = []

# Critério de corte acadêmico
LIMITE_PERDA_DADOS = 10.0  # Máximo de 10% de NaNs permitidos
LIMITE_VARIABILIDADE = 0.01  # Coeficiente de variação mínimo (1%)

for col in df_numerico.columns:
    serie = df_numerico[col].dropna()

    # 1. Filtra colunas com muita perda de dados (sensores que falharam)
    pct_nulos = (df_numerico[col].isna().sum() / len(df)) * 100
    if pct_nulos > LIMITE_PERDA_DADOS:
        colunas_quase_vazias.append(col)
        continue

    # 2. Filtra colunas constantes (sensores que não mudaram nada no teste)
    media = serie.mean()
    desvio = serie.std()

    if media == 0 or pd.isna(desvio) or desvio == 0:
        colunas_constantes.append(col)
        continue

    # Coeficiente de variação (Desvio Padrão / Média)
    cv = desvio / abs(media)

    if cv < LIMITE_VARIABILIDADE:
        colunas_constantes.append(col)
    else:
        colunas_relevantes.append(
            {"Coluna": col, "Média": media, "Desvio": desvio, "CV (%)": round(cv * 100, 2)}
        )

# Converte o resultado em um DataFrame estruturado
df_relevancia = pd.DataFrame(colunas_relevantes).sort_values(
    "CV (%)", ascending=False
)

print(f"--- Relatório de Relevância Científica ({Path(caminho_arquivo).name}) ---")
print(f"✅ Colunas Relevantes detectadas (Variam no teste): {len(df_relevancia)}")
print(f"❌ Colunas Inúteis/Constantes (Não variam): {len(colunas_constantes)}")
print(f"❌ Colunas Descartadas (Muitos NaNs): {len(colunas_quase_vazias)}\n")

print("🔝 Top 10 Colunas mais dinâmicas (Maior impacto visual em gráficos):")
display(df_relevancia.head(10))


--- Relatório de Relevância Científica (maqA_rodada_01.CSV) ---
✅ Colunas Relevantes detectadas (Variam no teste): 162
❌ Colunas Inúteis/Constantes (Não variam): 68
❌ Colunas Descartadas (Muitos NaNs): 1

🔝 Top 10 Colunas mais dinâmicas (Maior impacto visual em gráficos):


,Coluna,Média,Desvio,CV (%)
112,P-core 1 T1 C0 Ocupação [%],0.070588,0.266375,377.37
54,P-core 1 T1 Utilização [%],0.135294,0.482106,356.34
25,P-core 1 T1 Relógio efetivo [MHz],3.411765,11.830674,346.76
52,P-core 0 T1 Utilização [%],0.147059,0.458418,311.72
23,P-core 0 T1 Relógio efetivo [MHz],3.852941,11.512445,298.80
29,P-core 3 T1 Relógio efetivo [MHz],54.370588,160.329223,294.88
127,E-core 5 C1 Ocupação [%],0.952941,2.764534,290.11
110,P-core 0 T1 C0 Ocupação [%],0.100000,0.269258,269.26
116,P-core 3 T1 C0 Ocupação [%],1.470588,3.918987,266.49
133,P-core 2 C6 Ocupação [%],0.241176,0.626557,259.79


In [57]:
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Definição de caminhos do projeto
DIRETORIO_RAW = RAIZ_PROJETO / "data" / "raw" / "machine_a"
DIRETORIO_PROCESSED = RAIZ_PROJETO / "data" / "processed" / "machine_a"

# Garante a criação física da pasta de destino em inglês antes de salvar
DIRETORIO_PROCESSED.mkdir(parents=True, exist_ok=True)

# Parâmetros matemáticos de relevância científica
LIMITE_PERDA_DADOS = 10.0    # Descarta colunas com mais de 10% de NaNs
LIMITE_VARIABILIDADE = 0.01  # Coeficiente de variação mínimo (1%)

print(f"--- Iniciando Processamento em Lote Corrigido: Máquina A ---")
print(f"Destino: {DIRETORIO_PROCESSED}\n")

# 2. Loop para processar as 20 rodadas
for r in range(1, 21):
    nome_arquivo = f"maqA_rodada_{r:02d}.CSV"
    caminho_entrada = DIRETORIO_RAW / nome_arquivo
    
    if not caminho_entrada.is_file():
        print(f"⚠️ Arquivo ausente: {nome_arquivo} (Pulado)")
        continue
        
    # CORREÇÃO: Adicionado decimal="," para forçar a conversão correta de decimais brasileiros (vírgula)
    df_bruto = pd.read_csv(
        caminho_entrada, 
        encoding="utf-8", 
        on_bad_lines='skip', 
        encoding_errors='replace',
        decimal=","
    )
    
    # Isola o timestamp temporal nativo para mantê-lo obrigatoriamente na tabela tratada
    colunas_preservadas = []
    for col_tempo in ["Time", "Date", "Hora"]:
        if col_tempo in df_bruto.columns:
            colunas_preservadas.append(col_tempo)
            
    # Agora o select_dtypes vai capturar TODOS os sensores corretamente como números
    df_numerico = df_bruto.select_dtypes(include=[np.number]).copy()
    colunas_relevantes_rodada = []
    
    # 3. Algoritmo de filtragem de dados por relevância
    for col in df_numerico.columns:
        if col in colunas_preservadas:
            continue
            
        serie = df_numerico[col].dropna()
        
        # Filtro A: Integridade das amostras
        pct_nulos = (df_numerico[col].isna().sum() / len(df_bruto)) * 100
        if pct_nulos > LIMITE_PERDA_DADOS:
            continue
            
        media = serie.mean()
        desvio = serie.std()
        
        # Filtro B: Exclusão de sensores fixos/constantes
        if media == 0 or pd.isna(desvio) or desvio == 0:
            continue
            
        # Filtro C: Coeficiente de variação (Variação real)
        cv = desvio / abs(media)
        if cv >= LIMITE_VARIABILIDADE:
            colunas_relevantes_rodada.append(col)
            
    # Une as colunas temporais com as colunas numéricas que passaram na triagem
    colunas_finais = colunas_preservadas + colunas_relevantes_rodada
    df_filtrado = df_bruto[colunas_finais].copy()
    
    # 4. Exportação dos arquivos higienizados
    caminho_saida = DIRETORIO_PROCESSED / nome_arquivo
    df_filtrado.to_csv(caminho_saida, index=False, encoding="utf-8")
    
    print(f"✅ Rodada {r:02d}: Reduzida de {df_bruto.shape} para {df_filtrado.shape} colunas relevantes -> Salva.")

print(f"\n🚀 Processamento concluído com sucesso nas 20 rodadas.")


--- Iniciando Processamento em Lote Corrigido: Máquina A ---
Destino: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\data\processed\machine_a

✅ Rodada 01: Reduzida de (17, 312) para (17, 30) colunas relevantes -> Salva.
✅ Rodada 02: Reduzida de (30, 384) para (30, 32) colunas relevantes -> Salva.
✅ Rodada 03: Reduzida de (314, 388) para (314, 2) colunas relevantes -> Salva.
✅ Rodada 04: Reduzida de (300, 387) para (300, 42) colunas relevantes -> Salva.
✅ Rodada 05: Reduzida de (315, 388) para (315, 2) colunas relevantes -> Salva.
✅ Rodada 06: Reduzida de (302, 388) para (302, 2) colunas relevantes -> Salva.
✅ Rodada 07: Reduzida de (308, 388) para (308, 2) colunas relevantes -> Salva.
✅ Rodada 08: Reduzida de (306, 388) para (306, 2) colunas relevantes -> Salva.
✅ Rodada 09: Reduzida de (315, 388) para (315, 2) colunas relevantes -> Salva.
✅ Rodada 10: Reduzida de (313, 388) para (313, 2) colunas relevantes -> Salva.
✅ Rodada 11: Reduzida de (309, 388) para (309, 2) colunas relevante

In [58]:
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Definição de caminhos do projeto
DIRETORIO_RAW = RAIZ_PROJETO / "data" / "raw" / "machine_a"
DIRETORIO_PROCESSED = RAIZ_PROJETO / "data" / "processed" / "machine_a"

DIRETORIO_PROCESSED.mkdir(parents=True, exist_ok=True)

LIMITE_PERDA_DADOS = 10.0    
LIMITE_VARIABILIDADE = 0.01  

print(f"--- Iniciando Processamento Adaptativo: Máquina A ---")
print(f"Destino: {DIRETORIO_PROCESSED}\n")

for r in range(1, 21):
    nome_arquivo = f"maqA_rodada_{r:02d}.CSV"
    caminho_entrada = DIRETORIO_RAW / nome_arquivo
    
    if not caminho_entrada.is_file():
        print(f"⚠️ Arquivo ausente: {nome_arquivo} (Pulado)")
        continue
        
    # TENTATIVA 1: Tenta ler com o padrão de vírgula como separador
    df_bruto = pd.read_csv(
        caminho_entrada, 
        encoding="utf-8", 
        on_bad_lines='skip', 
        encoding_errors='replace',
        sep=",",
        decimal="."
    )
    
    # SEGUNDA VERIFICAÇÃO: Se o Pandas leu tudo como 1 única coluna, o arquivo usa ponto e vírgula (padrão BR)
    if df_bruto.shape[1] <= 2:
        df_bruto = pd.read_csv(
            caminho_entrada, 
            encoding="utf-8", 
            on_bad_lines='skip', 
            encoding_errors='replace',
            sep=";",
            decimal=","
        )
        
    # Isolamento de colunas temporais
    colunas_preservadas = []
    for col_tempo in ["Time", "Date", "Hora"]:
        # Varredura parcial para capturar colunas mesmo com espaços extras do HWInfo
        col_encontrada = [c for c in df_bruto.columns if col_tempo.lower() in c.lower()]
        if col_encontrada:
            colunas_preservadas.extend(col_encontrada)
            
    df_numerico = df_bruto.select_dtypes(include=[np.number]).copy()
    colunas_relevantes_rodada = []
    
    for col in df_numerico.columns:
        if col in colunas_preservadas:
            continue
            
        serie = df_numerico[col].dropna()
        pct_nulos = (df_numerico[col].isna().sum() / len(df_bruto)) * 100
        if pct_nulos > LIMITE_PERDA_DADOS:
            continue
            
        media = serie.mean()
        desvio = serie.std()
        
        if media == 0 or pd.isna(desvio) or desvio == 0:
            continue
            
        cv = desvio / abs(media)
        if cv >= LIMITE_VARIABILIDADE:
            colunas_relevantes_rodada.append(col)
            
    colunas_finais = list(set(colunas_preservadas + colunas_relevantes_rodada))
    df_filtrado = df_bruto[colunas_finais].copy()
    
    caminho_saida = DIRETORIO_PROCESSED / nome_arquivo
    df_filtrado.to_csv(caminho_saida, index=False, encoding="utf-8")
    
    print(f"✅ Rodada {r:02d}: Reduzida de {df_bruto.shape} para {df_filtrado.shape} colunas relevantes -> Salva.")

print(f"\n🚀 Processamento concluído com sucesso nas 20 rodadas.")


--- Iniciando Processamento Adaptativo: Máquina A ---
Destino: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\data\processed\machine_a

✅ Rodada 01: Reduzida de (17, 312) para (17, 164) colunas relevantes -> Salva.
✅ Rodada 02: Reduzida de (30, 384) para (30, 168) colunas relevantes -> Salva.
✅ Rodada 03: Reduzida de (314, 388) para (314, 2) colunas relevantes -> Salva.
✅ Rodada 04: Reduzida de (300, 387) para (300, 209) colunas relevantes -> Salva.
✅ Rodada 05: Reduzida de (315, 388) para (315, 2) colunas relevantes -> Salva.
✅ Rodada 06: Reduzida de (302, 388) para (302, 2) colunas relevantes -> Salva.
✅ Rodada 07: Reduzida de (308, 388) para (308, 2) colunas relevantes -> Salva.
✅ Rodada 08: Reduzida de (306, 388) para (306, 2) colunas relevantes -> Salva.
✅ Rodada 09: Reduzida de (315, 388) para (315, 2) colunas relevantes -> Salva.
✅ Rodada 10: Reduzida de (313, 388) para (313, 2) colunas relevantes -> Salva.
✅ Rodada 11: Reduzida de (309, 388) para (309, 2) colunas relevantes ->

In [59]:
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Definição de caminhos do projeto
DIRETORIO_RAW = RAIZ_PROJETO / "data" / "raw" / "machine_a"
DIRETORIO_PROCESSED = RAIZ_PROJETO / "data" / "processed" / "machine_a"
DIRETORIO_PROCESSED.mkdir(parents=True, exist_ok=True)

LIMITE_PERDA_DADOS = 10.0    
LIMITE_VARIABILIDADE = 0.01  

print(f"--- Iniciando Processamento Forçado (Fim do Bug das Rodadas Curtas) ---")
print(f"Destino: {DIRETORIO_PROCESSED}\n")

for r in range(1, 21):
    nome_arquivo = f"maqA_rodada_{r:02d}.CSV"
    caminho_entrada = DIRETORIO_RAW / nome_arquivo
    
    if not caminho_entrada.is_file():
        print(f"⚠️ Arquivo ausente: {nome_arquivo} (Pulado)")
        continue
        
    # Carrega mantendo o separador por vírgula nativo detectado
    df_bruto = pd.read_csv(
        caminho_entrada, 
        encoding="utf-8", 
        on_bad_lines='skip', 
        encoding_errors='replace',
        sep=","
    )
    
    # Isola colunas temporais
    colunas_preservadas = []
    for col_tempo in ["Time", "Date", "Hora"]:
        col_encontrada = [c for c in df_bruto.columns if col_tempo.lower() in c.lower()]
        if col_encontrada:
            colunas_preservadas.extend(col_encontrada)

    # CORREÇÃO CRÍTICA: Se a rodada tiver a assinatura estrutural problemática (388/389 colunas),
    # nós limpamos as strings trocando eventuais vírgulas remanescentes e forçando o tipo float.
    if df_bruto.shape[1] in:
        for col in df_bruto.columns:
            if col not in colunas_preservadas:
                # Transforma para string, limpa espaços, converte vírgula para ponto e força float
                df_bruto[col] = df_bruto[col].astype(str).str.replace(" ", "", regex=False)
                df_bruto[col] = pd.to_numeric(df_bruto[col].str.replace(",", ".", regex=False), errors='coerce')

    # Agora o select_dtypes funcionará perfeitamente em 100% das rodadas
    df_numerico = df_bruto.select_dtypes(include=[np.number]).copy()
    colunas_relevantes_rodada = []
    
    for col in df_numerico.columns:
        if col in colunas_preservadas:
            continue
            
        serie = df_numerico[col].dropna()
        pct_nulos = (df_numerico[col].isna().sum() / len(df_bruto)) * 100
        if pct_nulos > LIMITE_PERDA_DADOS:
            continue
            
        media = serie.mean()
        desvio = serie.std()
        
        if media == 0 or pd.isna(desvio) or desvio == 0:
            continue
            
        cv = desvio / abs(media)
        if cv >= LIMITE_VARIABILIDADE:
            colunas_relevantes_rodada.append(col)
            
    colunas_finais = list(set(colunas_preservadas + colunas_relevantes_rodada))
    df_filtrado = df_bruto[colunas_finais].copy()
    
    caminho_saida = DIRETORIO_PROCESSED / nome_arquivo
    df_filtrado.to_csv(caminho_saida, index=False, encoding="utf-8")
    
    print(f"✅ Rodada {r:02d}: Reduzida de {df_bruto.shape} para {df_filtrado.shape} colunas relevantes -> Salva.")

print(f"\n🚀 Processamento concluído. Todas as 20 rodadas possuem dados densos e estruturados.")



SyntaxError: invalid syntax (307365915.py, line 42)

## Etapa 2 — Análise dos resultados de Benchmark (Geekbench 6)

Calculamos, para cada máquina, a **média aritmética** e o **desvio padrão
amostral** dos scores *Single-Core* e *Multi-Core* ao longo das rodadas
disponíveis.

**Fórmulas utilizadas** (a transcrever no artigo, Seção de Metodologia):

$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i \qquad\qquad
s = \sqrt{\frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

O uso do desvio padrão **amostral** (denominador $n-1$, correção de Bessel)
é apropriado porque as 20 rodadas constituem uma **amostra** do
comportamento da máquina, não a população completa de execuções possíveis
(Hennessy e Patterson, 2018).


In [7]:
resultado_benchmark = bench.consolidar_benchmark_todas_maquinas(
    relatorios, mapa_nomes_exibicao=NOMES_EXIBICAO
) if relatorios else {"tabelas_individuais": {}, "scores_brutos": {}, "comparativo": pd.DataFrame()}

print("--- Estatísticas individuais por máquina ---")
for nome_maquina, tabela in resultado_benchmark["tabelas_individuais"].items():
    print(f"\n{NOMES_EXIBICAO.get(nome_maquina, nome_maquina)} ({nome_maquina}):")
    display(tabela)


[01:18:44] ERROR    | aoc_pipeline.benchmark | 'scores_maqE.txt': nenhuma rodada válida foi carregada.
--- Estatísticas individuais por máquina ---

Máquina A (machine_a):


,media,desvio_padrao,coef_variacao_pct,minimo,maximo,n_rodadas
Single_Core,2216.2,21.20,0.96,2139.0,2240.0,20.0
Multi_Core,8116.5,135.26,1.67,7767.0,8273.0,20.0



Máquina B (machine_b):


,media,desvio_padrao,coef_variacao_pct,minimo,maximo,n_rodadas
Single_Core,1851.95,91.09,4.92,1590.0,1910.0,20.0
Multi_Core,6107.70,155.71,2.55,5692.0,6300.0,20.0



Máquina C (machine_c):


,media,desvio_padrao,coef_variacao_pct,minimo,maximo,n_rodadas
Single_Core,932.0,95.94,10.29,705.0,987.0,20.0
Multi_Core,2415.5,321.82,13.32,1671.0,2653.0,20.0



Máquina D (machine_d):


,media,desvio_padrao,coef_variacao_pct,minimo,maximo,n_rodadas
Single_Core,1322.45,9.63,0.73,1291.0,1334.0,20.0
Multi_Core,2878.95,101.57,3.53,2580.0,2988.0,20.0



Máquina F (machine_f):


,media,desvio_padrao,coef_variacao_pct,minimo,maximo,n_rodadas
Single_Core,2828.3,21.82,0.77,2776.0,2849.0,20.0
Multi_Core,14241.0,85.17,0.60,14045.0,14414.0,20.0


### 📊 Gráfico: Análise de Dispersão e Estabilidade Estatística (Boxplot)

**Objetivo:** Avaliar a consistência do ambiente de testes e a repetibilidade experimental do benchmark ao longo das 20 rodadas consecutivas de estresse.

#### 🎯 Resultados Esperados e Interpretação Científica:
* <small>**Dispersão Reduzida (Boxes "achatados"):** Indica um ambiente experimental de alta fidelidade e isolamento estável. Prova que o sistema operacional mitigou ruídos de processos em segundo plano (*background noise*) e interrupções espúrias.</small>
* <small>**Dispersão Ampla ou Presença de *Outliers* Inferiores:** Sugere instabilidade no determinismo do hardware. Pontos isolados muito abaixo da mediana evidenciam gargalos momentâneos causados por picos térmicos ou contenção de recursos do sistema operacional.</small>
* <small>**Utilização no Artigo:** Este gráfico valida o rigor metodológico. Ele fundamenta estatisticamente a escolha da média amostral como métrica central representativa do desempenho bruto das microarquiteturas avaliadas.</small>



In [8]:
# Exemplo de implementação usando os dados brutos de todas as rodadas
import matplotlib.pyplot as plt
import seaborn as sns

if resultado_benchmark["scores_brutos"]:
    # Converte o dicionário de scores brutos de volta para um DataFrame longo (tidy data)
    df_longo = pd.DataFrame([
        {"Máquina": NOMES_EXIBICAO.get(maq, maq), "Score": score, "Tipo": tipo}
        for maq, dados in resultado_benchmark["scores_brutos"].items()
        for tipo, lista_scores in dados.items()  # 'Single_Core' ou 'Multi_Core'
        for score in lista_scores
    ])
    
    # Filtra apenas o Multi-Core para o gráfico principal
    df_multi = df_longo[df_longo["Tipo"] == "Multi_Core"]
    
    plt.figure(figsize=(7, 4))
    # hue="Máquina" e legend=False elimina o aviso do Seaborn
    sns.boxplot(x="Máquina", y="Score", hue="Máquina", data=df_multi, palette="gist_gray", width=0.5, legend=False)
    plt.title("Variabilidade do Score Multi-Core ao Longo das Rodadas")
    plt.ylabel("Geekbench 6 Score")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(caminhos.resultados_figuras / "final_plots" / "fig_boxplot_variabilidade_multicore.png", dpi=300)
    plt.close() # plt.close() evita o UserWarning de canvas não-interativo

    print(f"Gráfico de Boxplot salvo em: {caminhos.resultados_figuras / 'final_plots' / 'fig_boxplot_variabilidade_multicore.png'}")
    


Gráfico de Boxplot salvo em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\final_plots\fig_boxplot_variabilidade_multicore.png


### 📊 Gráfico: Comparativo Sétrico de Desempenho: Single-Core vs. Multi-Core

**Objetivo:** Contrastar a capacidade bruta de processamento em cenários de execução estritamente sequencial (Single-Core) versus cenários de paralelismo massivo de threads (Multi-Core), utilizando intervalos de confiança baseados no desvio padrão amostral.

#### 🎯 Resultados Esperados e Interpretação Científica:
* <small>**Fator de Escalabilidade (*Speedup*):** A razão entre o score Multi-Core e Single-Core quantifica a eficiência do paralelismo da arquitetura. Sistemas com núcleos híbridos (P-cores e E-cores) apresentarão dinâmicas de escalabilidade não-lineares se comparados a arquiteturas homogêneas.</small>
* <small>**Intervalos de Confiança (Barras de Erro `yerr`):** Barras de erro imperceptíveis demonstram que as médias são estatisticamente sólidas. Variações expressivas nos desvios padrão indicam assimetria na distribuição de carga entre os núcleos de execução.</small>
* <small>**Utilização no Artigo:** É a figura central da seção de **Resultados e Discussão**. Fornece o veredito empírico sobre o poder computacional bruto, respondendo quais inovações microarquiteturais (frequência de relógio, IPC ou densidade de núcleos) trouxeram maior ganho real.</small>


In [9]:
df_comp = resultado_benchmark["comparativo"]
if not df_comp.empty:
    import numpy as np
    
    x = np.arange(len(df_comp["maquina"]))
    largura = 0.35
    
    fig, ax = plt.subplots(figsize=(7, 4))
    
    # Barra de Single-Core (Cinza Claro)
    ax.bar(x - largura/2, df_comp["Single_Core_media"], largura, 
           yerr=df_comp["Single_Core_desvio_padrao"], label='Single-Core',
           color='#cccccc', edgecolor='#1a1a1a', capsize=4)
    
    # Barra de Multi-Core (Cinza Escuro)
    ax.bar(x + largura/2, df_comp["Multi_Core_media"], largura, 
           yerr=df_comp["Multi_Core_desvio_padrao"], label='Multi-Core',
           color='#666666', edgecolor='#1a1a1a', capsize=4)
    
    ax.set_ylabel('Geekbench 6 Score')
    ax.set_title('Comparativo de Desempenho Computacional por Máquina')
    ax.set_xticks(x)
    ax.set_xticklabels(df_comp["maquina"])
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(caminhos.resultados_figuras / "final_plots" / "fig_comparativo_desempenho_sbc.png", dpi=300)
    #plt.show()
    plt.close() # plt.close() evita o UserWarning de canvas não-interativo

    print(f"Gráfico de Comparativo de Desempenho salvo em: {caminhos.resultados_figuras / 'final_plots' / 'fig_comparativo_desempenho_sbc.png'}")

Gráfico de Comparativo de Desempenho salvo em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\final_plots\fig_comparativo_desempenho_sbc.png


### 📊 Gráfico: Evolução Temporal Inter-Rodadas e Degradação de Performance

**Objetivo:** Mapear o comportamento dinâmico do hardware sob estresse térmico prolongado, monitorando a evolução do score ao longo das 20 iterações sequenciais.

#### 🎯 Resultados Esperados e Interpretação Científica:
* <small>**Curva Descendente em Degraus (*Thermal Throttling*):** Máquinas com queda acentuada de rendimento após as primeiras rodadas comprovam a atuação dos mecanismos de proteção térmica (*prochot*). O processador reduz agressivamente o clock para evitar danos físicos, degradando o desempenho sustentado.</small>
* <small>**Curva Linear Horizontal (Estabilidade Térmica):** Evidencia um ecossistema microarquitetural equilibrado e um sistema de refrigeração ativo (dissipação/fans) capaz de sustentar o TDP (*Thermal Design Power*) máximo continuamente sem saturação térmica.</small>
* <small>**Utilização no Artigo:** Enquadra-se na seção de **Análise de Eficiência e Telemetria**. Este gráfico desmistifica o desempenho "de pico" (comumente reportado por fabricantes) e expõe a verdadeira capacidade de computação sustentada sob regime de carga pesada (*sustained workload*).</small>


In [10]:
import matplotlib.pyplot as plt

if resultado_benchmark["scores_brutos"]:
    plt.figure(figsize=(7.5, 4.5))
    
    # Marcadores e estilos de linha distintos no padrão SBC para diferenciar as máquinas
    estilos_marcadores = ["o", "s", "^", "D", "v", "X"]
    estilos_linhas = ["-", "--", "-.", ":", "-", "--"]
    # Tons de cinza/preto para publicação científica
    cores_sbc = ["#000000", "#555555", "#888888", "#aaaaaa", "#333333", "#777777"]
    
    idx = 0
    for nome_maquina, dados in resultado_benchmark["scores_brutos"].items():
        # Extrai a lista de scores Multi-Core (geralmente possui 20 elementos ordenados)
        scores_multi = dados.get("Multi_Core", [])
        
        # Uso de .empty para avaliar objetos de dados do Pandas com segurança
        if hasattr(scores_multi, "empty"):
            if scores_multi.empty:
                continue
        elif not scores_multi:
            continue
            
        # Cria o eixo X dinamicamente com base no número de rodadas executadas (Ex: 1 a 20)
        rodadas = list(range(1, len(scores_multi) + 1))
        
        # Plota a linha da máquina com marcadores geométricos
        plt.plot(
            rodadas, 
            scores_multi, 
            label=NOMES_EXIBICAO.get(nome_maquina, nome_maquina),
            marker=estilos_marcadores[idx % len(estilos_marcadores)],
            linestyle=estilos_linhas[idx % len(estilos_linhas)],
            color=cores_sbc[idx % len(cores_sbc)],
            linewidth=1.2,
            markersize=5,
            markeredgecolor="#000000"
        )
        idx += 1

    # Configurações de layout científico (SBC)
    plt.title("Evolução do Desempenho Multi-Core ao Longo do Teste de Estresse")
    plt.xlabel("Número da Rodada (Execução Sequencial)")
    plt.ylabel("Geekbench 6 Score")
    
    # Garante que o eixo X mostre apenas números inteiros das rodadas
    plt.xticks(range(1, 21)) 
    
    # Posiciona a legenda de forma limpa
    plt.legend(loc="lower left", frameon=True, edgecolor="#cccccc")
    plt.grid(axis="both", linestyle=":", alpha=0.6, color="#aaaaaa")
    
    plt.tight_layout()
    
    # Salva automaticamente na pasta correta de análise de dados brutos/intermediários
    caminho_figura_3 = caminhos.resultados_figuras / "raw_plots" / "fig_evolucao_rodadas_multicore.png"
    plt.savefig(caminho_figura_3, dpi=300)
    #plt.show()
    plt.close() # CORREÇÃO: Suprime o aviso de ambiente sem interface gráfica
    
    print(f"Gráfico de evolução temporal salvo em: {caminho_figura_3}")


Gráfico de evolução temporal salvo em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\raw_plots\fig_evolucao_rodadas_multicore.png


In [11]:
# Tabela comparativa (uma linha por máquina) — pronta para a Tabela do artigo SBC
tabela_comparativa_benchmark = resultado_benchmark["comparativo"]
display(tabela_comparativa_benchmark)

if not tabela_comparativa_benchmark.empty:
    caminho_saida = caminhos.resultados_tabelas / "benchmark_comparativo.csv"
    tabela_comparativa_benchmark.to_csv(caminho_saida, index=False, encoding="utf-8")
    print(f"Tabela comparativa salva em: {caminho_saida}")


,maquina,pasta,Single_Core_media,Single_Core_desvio_padrao,Single_Core_coef_variacao_pct,Single_Core_minimo,Single_Core_maximo,Single_Core_n_rodadas,Multi_Core_media,Multi_Core_desvio_padrao,Multi_Core_coef_variacao_pct,Multi_Core_minimo,Multi_Core_maximo,Multi_Core_n_rodadas
0,Máquina A,machine_a,2216.20,21.20,0.96,2139.0,2240.0,20.0,8116.50,135.26,1.67,7767.0,8273.0,20.0
1,Máquina B,machine_b,1851.95,91.09,4.92,1590.0,1910.0,20.0,6107.70,155.71,2.55,5692.0,6300.0,20.0
2,Máquina C,machine_c,932.00,95.94,10.29,705.0,987.0,20.0,2415.50,321.82,13.32,1671.0,2653.0,20.0
3,Máquina D,machine_d,1322.45,9.63,0.73,1291.0,1334.0,20.0,2878.95,101.57,3.53,2580.0,2988.0,20.0
4,Máquina F,machine_f,2828.30,21.82,0.77,2776.0,2849.0,20.0,14241.00,85.17,0.60,14045.0,14414.0,20.0


Tabela comparativa salva em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\tables\benchmark_comparativo.csv


### 📊 Gráfico Comparativo Geral — Desempenho Multi-Core vs. Single-Core por Máquina

**Objetivo:** Sintetizar os resultados consolidados do benchmark para confrontar diretamente o desempenho de computação sequencial e paralela de todas as microarquiteturas em um único plano visual.

#### 🎯 Resultados Esperados e Interpretação Científica:
* <small>**Assimetria de Escalabilidade:** Máquinas de alto desempenho (Desktop/Servidor) devem apresentar barras Multi-Core massivamente superiores às de Single-Core. Laptops ou mini-PCs mostrarão uma razão menor devido a limitações de dissipação térmica (TDP restrito) que limitam o turbo de múltiplos núcleos simultâneos.</small>
* <small>**Sobreposição de Desvios (Barras de Erro):** Se as barras de erro de duas máquinas diferentes se sobrepuserem visualmente, significa que, estatisticamente, o desempenho delas é equivalente naquela métrica, mesmo que as médias numéricas sejam ligeiramente diferentes.</small>
* <small>**Utilização no Artigo:** É a figura principal de benchmark do artigo. Ela traduz visualmente a tabela `.csv` exportada, permitindo correlacionar os pilares de IPC (Instructions Per Cycle), contagem de núcleos físicos e frequências de operação discutidos no texto.</small>


In [12]:
import matplotlib.pyplot as plt
import numpy as np

# Só executa se o DataFrame comparativo tiver dados
if not tabela_comparativa_benchmark.empty:
    df_plot = tabela_comparativa_benchmark.copy()
    
    # Configurações de dimensões e posições das barras agrupadas
    posicoes = np.arange(len(df_plot["maquina"]))
    largura_barra = 0.35
    
    # Inicializa a figura no padrão SBC (proporção limpa para colunas de artigos)
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    
    # 1. Plota as barras de Single-Core (Cinza Claro) com o desvio padrão amostral
    barras_single = ax.bar(
        posicoes - largura_barra/2, 
        df_plot["Single_Core_media"], 
        largura_barra, 
        yerr=df_plot["Single_Core_desvio_padrao"], 
        label="Single-Core",
        color="#cccccc", 
        edgecolor="#1a1a1a", 
        linewidth=0.8,
        capsize=4,
        error_kw={"elinewidth": 1.0, "ecolor": "#1a1a1a"}
    )
    
    # 2. Plota as barras de Multi-Core (Cinza Escuro) com o desvio padrão amostral
    barras_multi = ax.bar(
        posicoes + largura_barra/2, 
        df_plot["Multi_Core_media"], 
        largura_barra, 
        yerr=df_plot["Multi_Core_desvio_padrao"], 
        label="Multi-Core",
        color="#666666", 
        edgecolor="#1a1a1a", 
        linewidth=0.8,
        capsize=4,
        error_kw={"elinewidth": 1.0, "ecolor": "#1a1a1a"}
    )
    
    # Elementos textuais científicos (Evite títulos internos se o artigo usar sublegendas \caption)
    ax.set_ylabel("Geekbench 6 Score (Pontuação)")
    ax.set_xlabel("Microarquitetura / Máquina Avaliada")
    ax.set_title("Comparativo Consolidado de Desempenho Computacional")
    
    # Ajusta os rótulos do Eixo X com os nomes reais das máquinas
    ax.set_xticks(posicoes)
    ax.set_xticklabels(df_plot["maquina"].tolist(), rotation=0)
    
    # Legenda e grade de fundo sutil
    ax.legend(loc="upper left", frameon=True, edgecolor="#cccccc")
    ax.grid(axis="y", linestyle="--", alpha=0.5, color="#aaaaaa")
    
    # Remove bordas desnecessárias (estilo clean)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    # Salva automaticamente na estrutura correta em inglês para plots finais
    caminho_grafico = caminhos.resultados_figuras / "final_plots" / "fig_comparativo_consolidado_benchmark.png"
    plt.savefig(caminho_grafico, dpi=300)
    plt.close()
    
    print(f"Gráfico comparativo consolidado salvo em: {caminho_grafico}")


Gráfico comparativo consolidado salvo em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\final_plots\fig_comparativo_consolidado_benchmark.png


## Etapa 3 — Análise da Telemetria de Hardware (HWiNFO64)

Cada rodada gerou um arquivo `.csv` com **múltiplas amostras no tempo**
(série temporal). Para cada rodada, calculamos métricas-resumo (média e
máximo) das colunas críticas definidas pelo grupo: clock de núcleo, uso de
CPU/GPU, temperatura, potência e indicadores de *throttling* térmico.

Essas 20 (ou menos) linhas-resumo por máquina são então consolidadas em
estatísticas **ao nível de máquina** (média e desvio padrão **entre
rodadas**), que alimentam os gráficos comparativos da Etapa 5.


In [13]:
resultado_telemetria = tele.consolidar_telemetria_todas_maquinas(relatorios) if relatorios else {}

print("--- Estatísticas de telemetria por máquina (entre rodadas) ---")
estatisticas_telemetria = {}
for nome_maquina, df_consolidado in resultado_telemetria.items():
    if df_consolidado.empty:
        print(f"\n{nome_maquina}: nenhuma rodada de telemetria pôde ser processada.")
        continue
    stats = tele.calcular_estatisticas_telemetria(df_consolidado)
    estatisticas_telemetria[nome_maquina] = stats
    print(f"\n{NOMES_EXIBICAO.get(nome_maquina, nome_maquina)} ({nome_maquina}) "
          f"— {len(df_consolidado)} rodada(s) processada(s):")
    display(stats)



--- DEBUG: maqA_rodada_01.CSV ---
Linhas antes da limpeza (df_critico): 17
Colunas encontradas: ['clock_nucleo_mhz', 'clock_efetivo_nucleo_mhz', 'clock_gpu_mhz', 'uso_cpu_pct', 'carga_nucleo_gpu_pct', 'carga_memoria_fisica_pct', 'uso_memoria_gpu_pct', 'temp_cpu_c', 'temp_nucleo_max_c', 'temp_gpu_c', 'throttling_termico_nucleo', 'limite_termico_gpu_c', 'potencia_cpu_w', 'potencia_linhas_gpu_w', 'limite_pl1_atingido', 'limite_desempenho_termico', 'clock_memoria_mhz', 'taxa_leitura_mb_s', 'taxa_gravacao_mb_s']
Amostra dos dados antes da limpeza:
   clock_nucleo_mhz  clock_efetivo_nucleo_mhz  clock_gpu_mhz  uso_cpu_pct  carga_nucleo_gpu_pct  carga_memoria_fisica_pct  uso_memoria_gpu_pct  temp_cpu_c  \
0               NaN                       NaN          300.0          3.3                   0.8                       NaN                  NaN         NaN   
1               NaN                       NaN          300.0         10.1                   5.7                       NaN             

,media,desvio_padrao,coef_variacao_pct,n_rodadas
metrica,,,,
clock_gpu_mhz_media,295.366,88.225,29.87,20
clock_gpu_mhz_max,630.500,559.403,88.72,20
uso_cpu_pct_media,18.380,2.214,12.04,20
uso_cpu_pct_max,91.715,23.590,25.72,20
carga_nucleo_gpu_pct_media,1.232,0.987,80.10,20
carga_nucleo_gpu_pct_max,15.095,3.765,24.94,20
limite_pl1_atingido_pct_tempo,0.000,0.000,NaN,20
taxa_leitura_mb_s_media,5.090,3.094,60.79,20
taxa_leitura_mb_s_max,364.923,131.936,36.15,20



Máquina B (machine_b) — 20 rodada(s) processada(s):


,media,desvio_padrao,coef_variacao_pct,n_rodadas
metrica,,,,
clock_gpu_mhz_media,300.284,1.272,0.42,20
clock_gpu_mhz_max,302.500,11.180,3.70,20
uso_cpu_pct_media,18.525,1.489,8.04,20
uso_cpu_pct_max,97.420,2.837,2.91,20
carga_nucleo_gpu_pct_media,1.148,0.038,3.32,20
carga_nucleo_gpu_pct_max,10.135,1.343,13.25,20
limite_pl1_atingido_pct_tempo,2.122,1.697,79.99,20
taxa_leitura_mb_s_media,0.637,1.013,159.11,20
taxa_leitura_mb_s_max,16.650,19.989,120.06,20



Máquina C (machine_c) — 20 rodada(s) processada(s):


,media,desvio_padrao,coef_variacao_pct,n_rodadas
metrica,,,,
clock_gpu_mhz_media,200.153,0.684,0.34,20
clock_gpu_mhz_max,225.550,114.263,50.66,20
uso_cpu_pct_media,26.903,6.308,23.45,20
uso_cpu_pct_max,95.360,17.679,18.54,20
taxa_leitura_mb_s_media,2.123,4.725,222.53,20
taxa_leitura_mb_s_max,63.693,80.922,127.05,20



Máquina D (machine_d) — 20 rodada(s) processada(s):


,media,desvio_padrao,coef_variacao_pct,n_rodadas
metrica,,,,
clock_gpu_mhz_media,183.146,77.579,42.36,20
clock_gpu_mhz_max,312.230,150.072,48.06,20
uso_cpu_pct_media,29.216,4.303,14.73,20
uso_cpu_pct_max,95.740,19.051,19.90,20
limite_pl1_atingido_pct_tempo,17.906,5.915,33.03,20
taxa_leitura_mb_s_media,1.107,2.234,201.89,20
taxa_leitura_mb_s_max,23.922,27.593,115.35,20



Máquina E (machine_e) — 20 rodada(s) processada(s):


,media,desvio_padrao,coef_variacao_pct,n_rodadas
metrica,,,,
uso_cpu_pct_media,17.266,1.227,7.11,20
uso_cpu_pct_max,100.000,0.000,0.00,20
taxa_leitura_mb_s_media,0.696,1.236,177.77,20
taxa_leitura_mb_s_max,53.414,78.318,146.62,20


### 📊 Gráfico de Dispersão: Correlação entre Carga de CPU e Clock da GPU

**Objetivo:** Analisar a concorrência e o comportamento dinâmico de interdependência entre o processador principal e o subsistema gráfico sob regime de benchmark.

#### 🎯 Resultados Esperados e Interpretação Científica:
* <small>**Concentração de Pontos em Clocks Fixos:** Indica uma política de escalonamento de frequência conservadora ou um limite arquitetural rígido na GPU, operando em estados discretos (ex: 300 MHz ou 350 MHz) independentemente da flutuação da CPU.</small>
* <small>**Dispersão Inversa (Carga alta de CPU = Clock baixo de GPU):** Evidencia restrição de energia compartilhada (*Power Plane Budgeting*), comum em processadores mobile onde a CPU canibaliza o consumo da iGPU.</small>
* <small>**Utilização no Artigo:** Enriquece a análise de co-design de hardware, provando empiricamente como a carga do Geekbench distribui as exigências operacionais entre as diferentes unidades de computação.</small>


In [14]:
import matplotlib.pyplot as plt

# Garante que existem dados consolidados da telemetria
if resultado_telemetria:
    plt.figure(figsize=(7, 4.5))
    
    estilos_marcadores = ["o", "s", "^", "D"]
    cores_sbc = ["#333333", "#666666", "#999999", "#1a1a1a"]
    
    idx = 0
    for nome_maquina, df_maquina in resultado_telemetria.items():
        if df_maquina.empty or "uso_cpu_pct" not in df_maquina.columns or "clock_gpu_mhz" not in df_maquina.columns:
            continue
            
        # Filtra valores nulos para evitar falhas no plot
        df_valido = df_maquina.dropna(subset=["uso_cpu_pct", "clock_gpu_mhz"])
        
        if df_valido.empty:
            continue
            
        plt.scatter(
            df_valido["uso_cpu_pct"], 
            df_valido["clock_gpu_mhz"],
            label=NOMES_EXIBICAO.get(nome_maquina, nome_maquina),
            alpha=0.6,
            edgecolors="#1a1a1a",
            linewidths=0.5,
            color=cores_sbc[idx % len(cores_sbc)],
            marker=estilos_marcadores[idx % len(estilos_marcadores)],
            s=25
        )
        idx += 1
        
    plt.title("Perfil de Co-Processamento: Carga de CPU vs. Frequência da GPU")
    plt.xlabel("Utilização Total da CPU (%)")
    plt.ylabel("Clock do Núcleo da GPU (MHz)")
    plt.legend(loc="upper right", frameon=True, edgecolor="#cccccc")
    plt.grid(True, linestyle=":", alpha=0.6, color="#aaaaaa")
    plt.tight_layout()
    
    caminho_figura = caminhos.resultados_figuras / "raw_plots" / "fig_dispersao_cpu_gpu.png"
    plt.savefig(caminho_figura, dpi=300)
    plt.close()
    print(f"Gráfico de dispersão salvo em: {caminho_figura}")


C:\Users\pamel\AppData\Local\Temp\ipykernel_2696\1823413951.py:37: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(loc="upper right", frameon=True, edgecolor="#cccccc")


Gráfico de dispersão salvo em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\raw_plots\fig_dispersao_cpu_gpu.png


### 📊 Gráfico de Distribuição Cumulativa (CDF): Perfil de Carga da CPU

**Objetivo:** Analisar estatisticamente o perfil de estresse térmico e de processamento da CPU, normalizando a variação temporal das rodadas.

#### 🎯 Resultados Esperados e Interpretação Científica:
* <small>**Curva Deslocada para a Direita:** Demonstra alta densidade de computação sustentada. Prova que o benchmark manteve o processador sob estresse máximo durante quase toda a execução.</small>
* <small>**Curva com Degrau em Cargas Baixas:** Revela gargalos de E/S (Input/Output) ou sincronização de threads, onde a CPU permaneceu aguardando operações de disco ou memória externa.</small>
* <small>**Utilização no Artigo:** Valida o comportamento dinâmico do algoritmo do benchmark, demonstrando se a ferramenta estressa igualmente arquiteturas de escalas distintas.</small>


In [15]:
for k, v in resultado_telemetria.items():
    print(f"Chave: {k} | Tipo do valor: {type(v)}")
    if hasattr(v, 'columns'):
        print("Colunas encontradas:", list(v.columns))
    elif isinstance(v, dict):
        print("Chaves internas:", list(v.keys()))
    break


Chave: machine_a | Tipo do valor: <class 'pandas.DataFrame'>
Colunas encontradas: ['rodada', 'clock_nucleo_mhz_media', 'clock_nucleo_mhz_max', 'clock_efetivo_nucleo_mhz_media', 'clock_efetivo_nucleo_mhz_max', 'clock_gpu_mhz_media', 'clock_gpu_mhz_max', 'uso_cpu_pct_media', 'uso_cpu_pct_max', 'carga_nucleo_gpu_pct_media', 'carga_nucleo_gpu_pct_max', 'carga_memoria_fisica_pct_media', 'carga_memoria_fisica_pct_max', 'uso_memoria_gpu_pct_media', 'uso_memoria_gpu_pct_max', 'temp_cpu_c_media', 'temp_cpu_c_max', 'temp_nucleo_max_c_media', 'temp_nucleo_max_c_max', 'temp_gpu_c_media', 'temp_gpu_c_max', 'throttling_termico_nucleo_pct_tempo', 'limite_termico_gpu_c_media', 'limite_termico_gpu_c_max', 'potencia_cpu_w_media', 'potencia_cpu_w_max', 'potencia_linhas_gpu_w_media', 'potencia_linhas_gpu_w_max', 'limite_pl1_atingido_pct_tempo', 'limite_desempenho_termico_pct_tempo', 'clock_memoria_mhz_media', 'clock_memoria_mhz_max', 'taxa_leitura_mb_s_media', 'taxa_leitura_mb_s_max', 'taxa_gravacao_mb_s_

In [16]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if resultado_telemetria:
    # Inicializa a figura explicitamente para evitar gráficos em branco
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    
    estilos_marcadores = ["o", "s", "^", "D", "v", "X"]
    estilos_linhas = ["-", "--", "-.", ":", "-", "--"]
    cores_sbc = ["#000000", "#555555", "#888888", "#aaaaaa", "#333333", "#777777"]
    
    idx = 0
    dados_plotados = False
    
    for nome_maquina, df_maquina in resultado_telemetria.items():
        if not isinstance(df_maquina, pd.DataFrame) or df_maquina.empty:
            continue
            
        coluna_alvo = "uso_cpu_pct_media"
        if coluna_alvo not in df_maquina.columns:
            continue
            
        # Garante a ordenação pelas rodadas (1 a 20)
        df_ordenado = df_maquina.sort_values("rodada")
        
        # Remove NaNs se houver alguma rodada corrompida
        df_valido = df_ordenado.dropna(subset=[coluna_alvo, "rodada"])
        
        if df_valido.empty:
            continue
            
        # Plota a linha da respectiva máquina
        ax.plot(
            df_valido["rodada"].astype(int), 
            df_valido[coluna_alvo], 
            label=NOMES_EXIBICAO.get(nome_maquina, nome_maquina),
            marker=estilos_marcadores[idx % len(estilos_marcadores)],
            linestyle=estilos_linhas[idx % len(estilos_linhas)],
            color=cores_sbc[idx % len(cores_sbc)],
            linewidth=1.2,
            markersize=5,
            markeredgecolor="#000000"
        )
        idx += 1
        dados_plotados = True
        
    if dados_plotados:
        ax.set_title("Estabilidade da Carga de Trabalho: Uso de CPU Médio por Rodada")
        ax.set_xlabel("Número da Rodada (Execução Sequencial)")
        ax.set_ylabel("Uso Médio da CPU (%)")
        
        # Força o eixo X a exibir de 1 a 20 de forma limpa
        ax.set_xticks(range(1, 21))
        ax.set_ylim(0, 100) # Mantém a escala até 100%
        
        # Move a legenda para o topo esquerdo (área vazia do gráfico)
        # ax.legend( loc="upper left", frameon=True, edgecolor="#cccccc", facecolor="white", framealpha=0.9)
        # ALTERNATIVA: Legenda horizontal centralizada abaixo do gráfico
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=True, edgecolor="#cccccc")

        ax.grid(axis="both", linestyle=":", alpha=0.6, color="#aaaaaa")
        
        # Remove bordas desnecessárias para o padrão SBC
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        
        # Salva o arquivo final
        caminho_figura = caminhos.resultados_figuras / "final_plots" / "fig_uso_cpu_por_rodada.png"
        plt.savefig(caminho_figura, dpi=300)
        plt.close(fig) # Fecha a figura da memória de forma limpa
        print(f"Gráfico de uso de CPU gerado com sucesso em: {caminho_figura}")
    else:
        plt.close(fig)
        print("Aviso: Nenhuma coluna de uso de CPU válida foi localizada nos DataFrames de telemetria.")


Gráfico de uso de CPU gerado com sucesso em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\final_plots\fig_uso_cpu_por_rodada.png


In [17]:
# Percentual médio de tempo em throttling térmico, por máquina
print("--- Percentual médio de tempo em Thermal Throttling, por máquina ---")
for nome_maquina, df_consolidado in resultado_telemetria.items():
    pct = tele.calcular_percentual_throttling(df_consolidado)
    nome_exib = NOMES_EXIBICAO.get(nome_maquina, nome_maquina)
    if pct is None:
        print(f"{nome_exib}: dado indisponível.")
    else:
        print(f"{nome_exib}: {pct:.2f}% do tempo de execução em throttling térmico.")


--- Percentual médio de tempo em Thermal Throttling, por máquina ---
Máquina A: dado indisponível.
Máquina B: dado indisponível.
Máquina C: dado indisponível.
Máquina D: dado indisponível.
Máquina E: dado indisponível.


In [18]:
# Salva, para cada máquina, a tabela "1 linha por rodada" (útil para auditoria/anexo)
for nome_maquina, df_consolidado in resultado_telemetria.items():
    if df_consolidado.empty:
        continue
    caminho_saida = caminhos.dados_processed / f"{nome_maquina}_telemetria_por_rodada.csv"
    df_consolidado.to_csv(caminho_saida, index=False, encoding="utf-8")
print("Tabelas de telemetria por rodada salvas em data/processed/.")


Tabelas de telemetria por rodada salvas em data/processed/.


### 📊 Gráfico de Correlação Microarquitetural:Frequência Efetiva vs. Pico Térmico

**Objetivo:** Investigar o impacto da temperatura máxima alcançada sobre o clock efetivo do processador ao longo das rodadas consecutivas de estresse.

#### 🎯 Resultados Esperados e Interpretação Científica:
* <small>**Agrupamento no Topo Esquerdo (Alta Performance/Frio):** Representa o comportamento ideal de sistemas de alta eficiência térmica (Desktop/Watercooler), mantendo clocks máximos sem elevação crítica de temperatura.</small>
* <small>**Dispersão Descendente para a Direita (*Throttling* Ativo):** Revela a perda severa de desempenho microarquitetural. Conforme a temperatura máxima sobe nas rodadas sucessivas, o hardware é forçado a derrubar a frequência de operação.</small>
* <small>**Utilização no Artigo:** Este gráfico fornece o diagnóstico físico do comportamento observado nos benchmarks, correlacionando de forma direta as causas térmicas com os efeitos de variação nos scores.</small>


In [19]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

if resultado_telemetria:
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    
    estilos_marcadores = ["o", "s", "^", "D", "v", "X"]
    cores_sbc = ["#000000", "#555555", "#888888", "#aaaaaa", "#333333", "#777777"]
    
    idx = 0
    dados_plotados = False
    
    for nome_maquina in resultado_telemetria.keys():
        # Reconstrói o caminho do arquivo gerado pelo seu loop
        caminho_csv = caminhos.dados_processed / f"{nome_maquina}_telemetria_por_rodada.csv"
        
        if not caminho_csv.is_file():
            continue
            
        df_rodadas = pd.read_csv(caminho_csv)
        
        # Mapeia as colunas necessárias para este cruzamento microarquitetural
        col_clock = "clock_efetivo_nucleo_mhz_media"
        col_temp = "temp_cpu_c_max"
        
        if col_clock not in df_rodadas.columns or col_temp not in df_rodadas.columns:
            continue
            
        df_valido = df_rodadas.dropna(subset=[col_clock, col_temp])
        if df_valido.empty:
            continue
            
        # Plota os pontos discretos das 20 rodadas da máquina
        ax.scatter(
            df_valido[col_temp],
            df_valido[col_clock],
            label=NOMES_EXIBICAO.get(nome_maquina, nome_maquina),
            marker=estilos_marcadores[idx % len(estilos_marcadores)],
            color=cores_sbc[idx % len(cores_sbc)],
            edgecolors="#1a1a1a",
            linewidths=0.6,
            s=40,
            alpha=0.85,
            zorder=3
        )
        idx += 1
        dados_plotados = True

    if dados_plotados:
        ax.set_title("Análise de Restrição Térmica: Clock Efetivo vs. Pico de Temperatura")
        ax.set_xlabel("Temperatura Máxima da CPU na Rodada (°C)")
        ax.set_ylabel("Frequência Efetiva Média (MHz)")
        
        # Posicionamento estratégico da legenda no canto inferior esquerdo (área fria comum)
        ax.legend(loc="lower left", frameon=True, edgecolor="#cccccc", facecolor="white", framealpha=0.9)
        ax.grid(axis="both", linestyle=":", alpha=0.6, color="#aaaaaa")
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        
        caminho_figura = caminhos.resultados_figuras / "raw_plots" / "fig_correlacao_clock_temperatura.png"
        plt.savefig(caminho_figura, dpi=300)
        plt.close(fig)
        print(f"Gráfico de correlação térmica gerado com sucesso em: {caminho_figura}")
    else:
        plt.close(fig)
        print("Aviso: Sensores de temperatura e clock concomitantes não encontrados para gerar a correlação.")


Aviso: Sensores de temperatura e clock concomitantes não encontrados para gerar a correlação.


### 📊 Gráfico: Evolução Dinâmica da Frequência da GPU por Rodada

**Objetivo:** Monitorar a estabilidade do clock médio do subsistema gráfico (iGPU/dGPU) ao longo das iterações sequenciais do benchmark.

#### 🎯 Resultados Esperados e Interpretação Científica:
* <small>**Linhas Horizontais Rígidas (Máquinas B, C, D):** Evidenciam que a frequência da GPU opera em um estado energético fixo basal, imune à flutuação de carga imposta na CPU.</small>
* <small>**Flutuação Inter-Rodadas (Máquina A):** Demonstra uma política de gerenciamento térmico/energético dinâmica (*DVFS*), adaptando a frequência conforme o orçamento térmico do encapsulamento oscila.</small>


In [20]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

if resultado_telemetria:
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    
    estilos_marcadores = ["o", "s", "^", "D", "v"]
    estilos_linhas = ["-", "--", "-.", ":", "-"]
    cores_sbc = ["#000000", "#555555", "#888888", "#aaaaaa", "#333333"]
    
    idx = 0
    dados_plotados = False
    
    for nome_maquina, df_maquina in resultado_telemetria.items():
        if not isinstance(df_maquina, pd.DataFrame) or df_maquina.empty:
            continue
            
        coluna_alvo = "clock_gpu_mhz_media"
        if coluna_alvo not in df_maquina.columns:
            continue
            
        df_valido = df_maquina.sort_values("rodada").dropna(subset=[coluna_alvo, "rodada"])
        if df_valido.empty:
            continue
            
        ax.plot(
            df_valido["rodada"].astype(int), 
            df_valido[coluna_alvo], 
            label=NOMES_EXIBICAO.get(nome_maquina, nome_maquina),
            marker=estilos_marcadores[idx % len(estilos_marcadores)],
            linestyle=estilos_linhas[idx % len(estilos_linhas)],
            color=cores_sbc[idx % len(cores_sbc)],
            linewidth=1.2,
            markersize=5,
            markeredgecolor="#000000"
        )
        idx += 1
        dados_plotados = True
        
    if dados_plotados:
        ax.set_title("Comportamento do Subsistema Gráfico: Frequência da GPU por Rodada")
        ax.set_xlabel("Número da Rodada (Execução Sequencial)")
        ax.set_ylabel("Clock Médio da GPU (MHz)")
        ax.set_xticks(range(1, 21))
        
        #ax.legend(loc="upper right", frameon=True, edgecolor="#cccccc", facecolor="white", framealpha=0.9)
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=True, edgecolor="#cccccc")
        ax.grid(axis="both", linestyle=":", alpha=0.6, color="#aaaaaa")
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        caminho_figura = caminhos.resultados_figuras / "raw_plots" / "fig_clock_gpu_por_rodada.png"
        plt.savefig(caminho_figura, dpi=300)
        plt.close(fig)
        print(f"Gráfico de clock da GPU salvo em: {caminho_figura}")
    else:
        plt.close(fig)
        print("Aviso: Dados de clock de GPU não localizados nas tabelas.")


Gráfico de clock da GPU salvo em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\raw_plots\fig_clock_gpu_por_rodada.png


### 📊 Gráfico: Perfil de Saturação de Carga (Uso Máximo de CPU)

**Objetivo:** Quantificar o nível máximo de paralelismo e estresse que a carga de trabalho do benchmark conseguiu impor sobre cada microarquitetura.

###### 🎯 Resultados Esperados e Interpretação Científica:
* <small>**Saturação de 100% (Máquinas B, C, D, E):** Prova que a ferramenta obteve sucesso em escalar threads paralelos e utilizar plenamente a capacidade dos núcleos de execução.</small>
* <small>**Subutilização Abaixo de 30% (Máquina A):** Evidencia um gargalo de concorrência ou uma limitação arquitetural severa, onde o benchmark não conseguiu engajar os núcleos adicionais, operando sob restrição.</small>


In [21]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

if resultado_telemetria:
    fig, ax = plt.subplots(figsize=(7, 4))
    
    categorias = []
    medias_max = []
    
    for nome_maquina, df_maquina in resultado_telemetria.items():
        if not isinstance(df_maquina, pd.DataFrame) or df_maquina.empty:
            continue
            
        if "uso_cpu_pct_max" in df_maquina.columns:
            categorias.append(NOMES_EXIBICAO.get(nome_maquina, nome_maquina))
            medias_max.append(df_maquina["uso_cpu_pct_max"].mean())
            
    if categorias:
        posicoes = np.arange(len(categorias))
        
        barras = ax.bar(
            posicoes, 
            medias_max, 
            color="#555555", 
            edgecolor="#1a1a1a", 
            linewidth=0.8,
            width=0.4
        )
        
        ax.bar_label(barras, fmt="%.1f%%", padding=3, fontsize=9)
        ax.set_title("Nível de Estresse de Carga: Média do Uso Máximo da CPU")
        ax.set_ylabel("Uso Máximo da CPU (%)")
        #ax.set_xlabel("Máquina")
        ax.set_xticks(posicoes)
        ax.set_xticklabels(categorias)
        ax.set_ylim(0, 110)
        
        ax.grid(axis="y", linestyle="--", alpha=0.5, color="#aaaaaa")
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        caminho_figura = caminhos.resultados_figuras / "final_plots" / "fig_saturacao_uso_cpu.png"
        plt.savefig(caminho_figura, dpi=300)
        plt.close(fig)
        print(f"Gráfico de saturação de CPU salvo em: {caminho_figura}")
    else:
        plt.close(fig)
        print("Aviso: Nenhuma coluna 'uso_cpu_pct_max' localizada.")


Gráfico de saturação de CPU salvo em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\final_plots\fig_saturacao_uso_cpu.png


## Etapa 4 — Eficiência Microarquitetural: Desempenho por Watt e IPC Relativo

Cruzamos os **scores médios** (Etapa 2) com a **potência média da CPU**
(Etapa 3) para estimar a eficiência energética de cada máquina:

$$\text{Desempenho por Watt} = \dfrac{\text{Score}_{\text{médio}}}{\overline{P}_{\text{CPU}} \ (\text{W})}$$

Também estimamos um indicador *proxy* de eficiência por ciclo de clock
(**não é o IPC real** — ver ressalva no docstring de
`calcular_ipc_relativo`, em `benchmark_analysis.py`):

$$\text{IPC}_{\text{relativo}} = \dfrac{\text{Score}_{\text{médio}}}{\overline{f}_{\text{efetiva}} \ (\text{MHz})}$$


In [22]:
linhas_eficiencia = []
for nome_maquina in resultado_benchmark["tabelas_individuais"]:
    tabela_bench = resultado_benchmark["tabelas_individuais"][nome_maquina]
    df_tele = resultado_telemetria.get(nome_maquina, pd.DataFrame())

    if df_tele.empty:
        continue

    stats_tele = tele.calcular_estatisticas_telemetria(df_tele)

    potencia_media = (
        stats_tele.loc["potencia_cpu_w_media", "media"]
        if "potencia_cpu_w_media" in stats_tele.index else np.nan
    )
    clock_efetivo_medio = (
        stats_tele.loc["clock_efetivo_nucleo_mhz_media", "media"]
        if "clock_efetivo_nucleo_mhz_media" in stats_tele.index else np.nan
    )
    score_multicore_medio = (
        tabela_bench.loc["Multi_Core", "media"] if "Multi_Core" in tabela_bench.index else np.nan
    )

    linhas_eficiencia.append({
        "maquina": NOMES_EXIBICAO.get(nome_maquina, nome_maquina),
        "pasta": nome_maquina,
        "score_multicore_medio": score_multicore_medio,
        "potencia_cpu_media_w": potencia_media,
        "clock_efetivo_medio_mhz": clock_efetivo_medio,
        "desempenho_por_watt": bench.calcular_desempenho_por_watt(score_multicore_medio, potencia_media),
        "ipc_relativo": bench.calcular_ipc_relativo(score_multicore_medio, clock_efetivo_medio),
        "pct_tempo_throttling": tele.calcular_percentual_throttling(df_tele),
    })

tabela_eficiencia = pd.DataFrame(linhas_eficiencia)
display(tabela_eficiencia)

if not tabela_eficiencia.empty:
    caminho_saida = caminhos.resultados_tabelas / "eficiencia_microarquitetural.csv"
    tabela_eficiencia.to_csv(caminho_saida, index=False, encoding="utf-8")
    print(f"Tabela de eficiência salva em: {caminho_saida}")


[01:22:08] WARNING  | aoc_pipeline.benchmark | Potência média inválida (nan); Desempenho/Watt não calculado.
[01:22:08] WARNING  | aoc_pipeline.benchmark | Potência média inválida (nan); Desempenho/Watt não calculado.
[01:22:08] WARNING  | aoc_pipeline.benchmark | Potência média inválida (nan); Desempenho/Watt não calculado.
[01:22:08] WARNING  | aoc_pipeline.benchmark | Potência média inválida (nan); Desempenho/Watt não calculado.


,maquina,pasta,score_multicore_medio,potencia_cpu_media_w,clock_efetivo_medio_mhz,desempenho_por_watt,ipc_relativo,pct_tempo_throttling
0,Máquina A,machine_a,8116.50,NaN,NaN,None,None,None
1,Máquina B,machine_b,6107.70,NaN,NaN,None,None,None
2,Máquina C,machine_c,2415.50,NaN,NaN,None,None,None
3,Máquina D,machine_d,2878.95,NaN,NaN,None,None,None


Tabela de eficiência salva em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\tables\eficiencia_microarquitetural.csv


### 📊 Gráfico de Eficiência Microarquitetural: IPC Relativo por Máquina

**Objetivo:** Comparar a eficiência de instruções por ciclo (IPC) relativo entre as diferentes arquiteturas, isolando o impacto da frequência bruta de relógio (clock).

#### 🎯 Resultados Esperados e Interpretação Científica:
* <small>**Maior IPC Relativo:** Indica uma microarquitetura mais moderna e eficiente, capaz de realizar mais trabalho computacional por ciclo de clock (melhor previsão de desvios, caches maiores ou pipelines mais largos).</small>
* <small>**Menor IPC Relativo:** Aponta para arquiteturas legadas ou severamente limitadas por latência de memória, dependendo exclusivamente de clocks elevados para entregar desempenho bruto.</small>


In [23]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if not tabela_eficiencia.empty:
    # Filtra e ordena para manter apenas dados válidos de IPC
    df_ipc = tabela_eficiencia.dropna(subset=["ipc_relativo"]).copy()
    df_ipc = df_ipc.sort_values("ipc_relativo", ascending=False)
    
    if not df_ipc.empty:
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        
        posicoes = np.arange(len(df_ipc["maquina"]))
        
        # Plota as barras em tom de cinza
        barras = ax.bar(
            posicoes, 
            df_ipc["ipc_relativo"], 
            color="#555555", 
            edgecolor="#1a1a1a", 
            linewidth=0.8,
            width=0.4
        )
        
        # Adiciona rótulos numéricos detalhados em cima de cada barra
        ax.bar_label(barras, fmt="%.4f", padding=3, fontsize=9)
        
        # Títulos e formatação acadêmica
        ax.set_title("Eficiência Microarquitetural: Métrica de IPC Relativo")
        ax.set_ylabel("IPC Relativo (Score / MHz)")
        ax.set_xticks(posicoes)
        ax.set_xticklabels(df_ipc["maquina"].tolist(), rotation=0)
        
        # Define folga no topo para o rótulo de dados não cortar
        max_val = df_ipc["ipc_relativo"].max()
        ax.set_ylim(0, max_val * 1.15)
        
        # Grid horizontal discreta
        ax.grid(axis="y", linestyle="--", alpha=0.5, color="#aaaaaa")
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        
        # Salva o arquivo na pasta correta de plots consolidados (final_plots)
        caminho_figura = caminhos.resultados_figuras / "final_plots" / "fig_ipc_relativo_maquinas.png"
        plt.savefig(caminho_figura, dpi=300)
        plt.close(fig)
        print(f"Gráfico de IPC Relativo salvo com sucesso em: {caminho_figura}")
    else:
        print("Aviso: Nenhuma máquina possui dados suficientes para calcular o IPC Relativo.")


Aviso: Nenhuma máquina possui dados suficientes para calcular o IPC Relativo.


## Etapa 5 — Geração dos Gráficos (padrão visual SBC)

Todos os gráficos seguem o padrão sóbrio exigido pela SBC: escala de
cinza, hachuras para diferenciar categorias sem depender de cor, e fonte
sans-serif. As figuras são salvas em `results/figuras/` em **300 DPI**,
prontas para `\includegraphics` no Overleaf.

> Lembre-se: a **legenda formal** (Helvetica 10pt negrito, posicionada
> ANTES da tabela / com a regra de centralização ou justificação) deve
> ser escrita diretamente no `.tex`, com o texto padrão ABNT
> *"Fonte: Os autores (2026)"* quando os dados forem da própria coleta do
> grupo.


In [24]:
# 5.1 — Score Multi-Core médio por máquina (Padrão SBC com Rótulos e Barras de Erro)
if not tabela_comparativa_benchmark.empty:
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    
    posicoes = np.arange(len(tabela_comparativa_benchmark["maquina"]))
    
    # Plota as barras mantendo o desvio padrão como erro amostral (yerr)
    barras = ax.bar(
        posicoes,
        tabela_comparativa_benchmark["Multi_Core_media"].tolist(),
        yerr=tabela_comparativa_benchmark["Multi_Core_desvio_padrao"].tolist(),
        capsize=4,
        color="#555555",
        edgecolor="#1a1a1a",
        linewidth=0.8,
        width=0.4,
        error_kw={"elinewidth": 1.0, "ecolor": "#1a1a1a"}
    )
    
    # ADICIONADO: Coloca os rótulos numéricos do score acima da barra de erro
    ax.bar_label(barras, fmt="%.1f", padding=5, fontsize=9, fontweight="bold")
    
    # Configurações textuais acadêmicas
    ax.set_title("Score Multi-Core Médio por Máquina (Geekbench 6)")
    ax.set_ylabel("Score Multi-Core")
    ax.set_xticks(posicoes)
    ax.set_xticklabels(tabela_comparativa_benchmark["maquina"].tolist(), rotation=0)
    
    # Dá uma folga no teto do eixo Y para a barra de erro e o rótulo não sumirem
    max_val = max(tabela_comparativa_benchmark["Multi_Core_media"].tolist())
    ax.set_ylim(0, max_val * 1.20)
    
    ax.grid(axis="y", linestyle="--", alpha=0.5, color="#aaaaaa")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    # Salva na pasta estruturada correta em inglês
    caminho_saida = caminhos.resultados_figuras / "final_plots" / "fig_score_multicore_por_maquina.png"
    plt.savefig(caminho_saida, dpi=300)
    plt.close(fig)
    print(f"Gráfico de Score Multi-Core gerado com sucesso em: {caminho_saida}")


Gráfico de Score Multi-Core gerado com sucesso em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\final_plots\fig_score_multicore_por_maquina.png


In [25]:
# 5.2 — Score Single-Core médio por máquina (Padrão SBC com Rótulos e Barras de Erro)
if not tabela_comparativa_benchmark.empty:
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    
    posicoes = np.arange(len(tabela_comparativa_benchmark["maquina"]))
    
    # Plota as barras mantendo o desvio padrão como erro amostral (yerr)
    barras = ax.bar(
        posicoes,
        tabela_comparativa_benchmark["Single_Core_media"].tolist(),
        yerr=tabela_comparativa_benchmark["Single_Core_desvio_padrao"].tolist(),
        capsize=4,
        color="#888888",  # Cinza médio para diferenciar visualmente do Multi-Core
        edgecolor="#1a1a1a",
        linewidth=0.8,
        width=0.4,
        error_kw={"elinewidth": 1.0, "ecolor": "#1a1a1a"}
    )
    
    # INCLUÍDO: Adiciona os rótulos de dados numéricos acima do desvio padrão
    ax.bar_label(barras, fmt="%.1f", padding=5, fontsize=9, fontweight="bold")
    
    # Configurações textuais científicas
    ax.set_title("Score Single-Core Médio por Máquina (Geekbench 6)")
    ax.set_ylabel("Score Single-Core")
    ax.set_xticks(posicoes)
    ax.set_xticklabels(tabela_comparativa_benchmark["maquina"].tolist(), rotation=0)
    
    # Define folga no topo do eixo Y para que o texto do rótulo não seja cortado
    max_val = max(tabela_comparativa_benchmark["Single_Core_media"].tolist())
    ax.set_ylim(0, max_val * 1.20)
    
    ax.grid(axis="y", linestyle="--", alpha=0.5, color="#aaaaaa")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    # Salva na pasta correta em inglês
    caminho_saida = caminhos.resultados_figuras / "final_plots" / "fig_score_singlecore_por_maquina.png"
    plt.savefig(caminho_saida, dpi=300)
    plt.close(fig)
    print(f"Gráfico de Score Single-Core gerado com sucesso em: {caminho_saida}")


Gráfico de Score Single-Core gerado com sucesso em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\final_plots\fig_score_singlecore_por_maquina.png


In [26]:
# 5.3 — Temperatura máxima de CPU por máquina (evidencia risco de throttling)
linhas_temp = []
for nome_maquina, stats in estatisticas_telemetria.items():
    if "temp_cpu_c_max" in stats.index:
        linhas_temp.append({
            "maquina": NOMES_EXIBICAO.get(nome_maquina, nome_maquina),
            "media": stats.loc["temp_cpu_c_max", "media"],
            "desvio": stats.loc["temp_cpu_c_max", "desvio_padrao"],
        })
df_temp = pd.DataFrame(linhas_temp)

if not df_temp.empty:
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    
    posicoes = np.arange(len(df_temp["maquina"]))
    
    # Desenha as barras com os desvios térmicos amostrais
    barras = ax.bar(
        posicoes,
        df_temp["media"].tolist(),
        yerr=df_temp["desvio"].tolist(),
        capsize=4,
        color="#777777", # Tom de cinza para dados de telemetria térmica
        edgecolor="#1a1a1a",
        linewidth=0.8,
        width=0.4,
        error_kw={"elinewidth": 1.0, "ecolor": "#1a1a1a"}
    )
    
    # Adiciona os rótulos de dados com uma casa decimal e o símbolo de graus Celsius
    ax.bar_label(barras, fmt="%.1f°C", padding=5, fontsize=9, fontweight="bold")
    
    # Configurações textuais científicas (unidade no eixo Y)
    ax.set_title("Temperatura Máxima de CPU por Rodada (Média entre Rodadas)")
    ax.set_ylabel("Temperatura (°C)")
    ax.set_xticks(posicoes)
    ax.set_xticklabels(df_temp["maquina"].tolist(), rotation=0)
    
    # Dá uma folga de 20% no teto do eixo Y para os rótulos e as barras de erro não sumirem
    max_val = max(df_temp["media"].tolist()) if df_temp["media"].tolist() else 100
    ax.set_ylim(0, max_val * 1.20)
    
    ax.grid(axis="y", linestyle="--", alpha=0.5, color="#aaaaaa")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    # Salvando na estrutura de pastas em inglês correta (raw_plots)
    caminho_saida = caminhos.resultados_figuras / "raw_plots" / "fig_temperatura_max_por_maquina.png"
    plt.savefig(caminho_saida, dpi=300)
    plt.close(fig)
    print(f"Gráfico de Temperatura Máxima gerado com sucesso em: {caminho_saida}")
else:
    print("Aviso: O gráfico de temperatura não foi gerado pois os sensores térmicos retornaram vazios (NaN).")


Aviso: O gráfico de temperatura não foi gerado pois os sensores térmicos retornaram vazios (NaN).


In [27]:
# 5.4 — Potência média da CPU por máquina
linhas_pot = []
for nome_maquina, stats in estatisticas_telemetria.items():
    if "potencia_cpu_w_media" in stats.index:
        linhas_pot.append({
            "maquina": NOMES_EXIBICAO.get(nome_maquina, nome_maquina),
            "media": stats.loc["potencia_cpu_w_media", "media"],
            "desvio": stats.loc["potencia_cpu_w_media", "desvio_padrao"],
        })
df_pot = pd.DataFrame(linhas_pot)

if not df_pot.empty:
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    
    posicoes = np.arange(len(df_pot["maquina"]))
    
    # Plota as barras de consumo com o desvio padrão amostral
    barras = ax.bar(
        posicoes,
        df_pot["media"].tolist(),
        yerr=df_pot["desvio"].tolist(),
        capsize=4,
        color="#666666", # Tom de cinza sólido para potência
        edgecolor="#1a1a1a",
        linewidth=0.8,
        width=0.4,
        error_kw={"elinewidth": 1.0, "ecolor": "#1a1a1a"}
    )
    
    # INCLUÍDO: Adiciona os rótulos de dados numéricos com o sufixo "W" (Watts) acima da barra de erro
    ax.bar_label(barras, fmt="%.1fW", padding=5, fontsize=9, fontweight="bold")
    
    # Configurações textuais científicas
    ax.set_title("Potência Média da CPU por Máquina")
    ax.set_ylabel("Potência (W)")
    ax.set_xticks(posicoes)
    ax.set_xticklabels(df_pot["maquina"].tolist(), rotation=0)
    
    # Define folga no topo do eixo Y para os rótulos textuais não cortarem
    max_val = max(df_pot["media"].tolist()) if df_pot["media"].tolist() else 50
    ax.set_ylim(0, max_val * 1.25)
    
    ax.grid(axis="y", linestyle="--", alpha=0.5, color="#aaaaaa")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    # CORREÇÃO: Salvando no diretório correto em inglês (raw_plots)
    caminho_saida = caminhos.resultados_figuras / "raw_plots" / "fig_potencia_cpu_por_maquina.png"
    plt.savefig(caminho_saida, dpi=300)
    plt.close(fig)
    print(f"Gráfico de potência gerado com sucesso em: {caminho_saida}")
else:
    print("Aviso: O gráfico de potência não foi gerado pois os sensores de Watts retornaram vazios (NaN) nas tabelas.")

Aviso: O gráfico de potência não foi gerado pois os sensores de Watts retornaram vazios (NaN) nas tabelas.


In [28]:
# 5.5 — Desempenho por Watt (eficiência energética) por máquina
if not tabela_eficiencia.empty:
    # Filtra para manter apenas máquinas que possuem dados válidos de eficiência
    tabela_filtrada = tabela_eficiencia.dropna(subset=["desempenho_por_watt"])

    if not tabela_eficiencia.empty:
        # Filtra para manter apenas máquinas que possuem dados válidos de eficiência
        tabela_filtrada = tabela_eficiencia.dropna(subset=["desempenho_por_watt"]).copy()

        if not tabela_filtrada.empty:
            fig, ax = plt.subplots(figsize=(7.5, 4.5))
            
            posicoes = np.arange(len(tabela_filtrada["maquina"]))
            
            # Plota as barras de eficiência energética (sem barra de erro, pois é métrica derivada)
            barras = ax.bar(
                posicoes, 
                tabela_filtrada["desempenho_por_watt"].tolist(), 
                color="#444444", # Tom de cinza escuro para diferenciar das métricas brutas
                edgecolor="#1a1a1a", 
                linewidth=0.8,
                width=0.4
            )
            
            # INCLUÍDO: Adiciona os rótulos numéricos de Score/Watt no topo de cada barra
            ax.bar_label(barras, fmt="%.2f", padding=3, fontsize=9, fontweight="bold")
            
            # Configurações textuais científicas
            ax.set_title("Eficiência Energética: Score Multi-Core por Watt")
            ax.set_ylabel("Eficiência (Score / Watt)")
            ax.set_xticks(posicoes)
            ax.set_xticklabels(tabela_filtrada["maquina"].tolist(), rotation=0)
            
            # Define folga no topo do eixo Y para o texto do rótulo não ser cortado
            max_val = max(tabela_filtrada["desempenho_por_watt"].tolist())
            ax.set_ylim(0, max_val * 1.20)
            
            # Grid horizontal discreta e remoção de bordas
            ax.grid(axis="y", linestyle="--", alpha=0.5, color="#aaaaaa")
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            
            plt.tight_layout()
            
            # CORREÇÃO: Salvando no diretório correto em inglês de gráficos consolidados (final_plots)
            caminho_saida = caminhos.resultados_figuras / "final_plots" / "fig_eficiencia_energetica_por_maquina.png"
            plt.savefig(caminho_saida, dpi=300)
            plt.close(fig)
            print(f"Gráfico de eficiência energética gerado com sucesso em: {caminho_saida}")
        else:
            print("Aviso: O gráfico de eficiência energética não foi gerado pois os sensores de Watts retornaram vazios (NaN) nas tabelas.")


Aviso: O gráfico de eficiência energética não foi gerado pois os sensores de Watts retornaram vazios (NaN) nas tabelas.


In [29]:
# 5.6 — Boxplot de variabilidade do Score Multi-Core entre rodadas
if resultado_benchmark.get("scores_brutos"):
    linhas_boxplot = []
    
    for nome_maquina, sub_dict in resultado_benchmark["scores_brutos"].items():
        # CORREÇÃO: Acessa a chave correta dentro do sub-dicionário de scores brutos
        scores_multi = sub_dict.get("Multi_Core", [])
        
        # Converte para lista nativa caso venha como Série do Pandas
        if hasattr(scores_multi, "tolist"):
            scores_multi = scores_multi.dropna().tolist()
            
        nome_exibicao = NOMES_EXIBICAO.get(nome_maquina, nome_maquina)
        for score in scores_multi:
            linhas_boxplot.append({"Máquina": nome_exibicao, "Score": score})
            
    df_box = pd.DataFrame(linhas_boxplot)

    if not df_box.empty:
        import seaborn as sns
        
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        
        # Plota o boxplot em tons de cinza científico (Padrão SBC)
        # O uso de hue e legend=False silencia os FutureWarnings do Seaborn
        sns.boxplot(
            x="Máquina", 
            y="Score", 
            hue="Máquina",
            data=df_box, 
            palette="gist_gray", 
            width=0.4, 
            ax=ax,
            legend=False,
            boxprops=dict(edgecolor='#1a1a1a', linewidth=0.8),
            whiskerprops=dict(color='#1a1a1a', linewidth=0.8),
            capprops=dict(color='#1a1a1a', linewidth=0.8),
            medianprops=dict(color='#000000', linewidth=1.2),
            flierprops=dict(marker='o', markerfacecolor='#666666', markersize=4)
        )
        
        # Configurações textuais rigorosas para o artigo
        ax.set_title("Variabilidade Estatística do Score Multi-Core (Estresse Contínuo)")
        ax.set_ylabel("Geekbench 6 Score (Multi-Core)")
        ax.set_xlabel("Microarquitetura / Máquina")
        
        # Adiciona uma grade horizontal discreta para facilitar a leitura das medianas
        ax.grid(axis='y', linestyle='--', alpha=0.5, color="#aaaaaa")
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        
        # Salva o arquivo na subpasta correta de resultados finais consolidados
        caminho_saida = caminhos.resultados_figuras / "final_plots" / "fig_boxplot_variabilidade_multicore.png"
        plt.savefig(caminho_saida, dpi=300)
        plt.close(fig)
        print(f"Gráfico Boxplot de variabilidade gerado com sucesso em: {caminho_saida}")
    else:
        print("Aviso: Dados de scores vazios para gerar o Boxplot.")


Gráfico Boxplot de variabilidade gerado com sucesso em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\figures\final_plots\fig_boxplot_variabilidade_multicore.png


In [30]:
# 5.7 — Dispersão: Clock efetivo médio (por rodada) x Score (IPC relativo)

for nome_maquina, df_tele in resultado_telemetria.items():
    dados_scores = resultado_benchmark["scores_brutos"].get(nome_maquina)
    
    # Valida se a máquina possui dados temporais válidos de clock e scores cadastrados
    if (df_tele.empty or 
        dados_scores is None or 
        "clock_efetivo_nucleo_mhz_media" not in df_tele.columns or 
        "Multi_Core" not in dados_scores):
        continue

    # 1. Alinha a Telemetria indexando pela rodada de forma limpa
    df_tele_idx = df_tele.copy()
    df_tele_idx["rodada"] = pd.to_numeric(df_tele_idx["rodada"], errors="coerce").astype(int)
    
    # 2. Reconstrói o DataFrame de scores brutos a partir das séries temporais indexadas
    series_multi = dados_scores["Multi_Core"]
    df_scores_idx = pd.DataFrame({
        "rodada_num": list(range(1, len(series_multi) + 1)),
        "Multi_Core_Score": series_multi
    })

    # Cruza os dois universos baseando-se no ID da rodada comum
    cruzado = df_tele_idx.merge(
        df_scores_idx, left_on="rodada", right_on="rodada_num", how="inner"
    )
    
    # Se a máquina não tiver dados em comum filtrados após o dropna, pula para evitar quebra
    cruzado_valido = cruzado.dropna(subset=["clock_efetivo_nucleo_mhz_media", "Multi_Core_Score"])
    if cruzado_valido.empty:
        continue

    # Inicializa a figura nativa do Matplotlib (Isola o Canvas da memória)
    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    
    # Plota a dispersão com marcadores geométricos pretos e cinza sutil (Padrão SBC)
    sc = ax.scatter(
        cruzado_valido["clock_efetivo_nucleo_mhz_media"],
        cruzado_valido["Multi_Core_Score"],
        color="#555555",
        edgecolors="#1a1a1a",
        linewidths=0.7,
        s=45,
        alpha=0.85,
        zorder=3
    )
    
    # Adiciona rótulos discretos internos ao lado de cada ponto identificando o número da rodada
    for _, linha in cruzado_valido.iterrows():
        ax.text(
            linha["clock_efetivo_nucleo_mhz_media"], 
            linha["Multi_Core_Score"], 
            f" R{int(linha['rodada'])}", 
            fontsize=7, 
            color="#333333",
            va='center', 
            ha='left'
        )

    # Configurações textuais acadêmicas rigorosas
    nome_exibicao = NOMES_EXIBICAO.get(nome_maquina, nome_maquina)
    ax.set_title(f"Sensibilidade de Frequência: Clock Efetivo vs. Score — {nome_exibicao}")
    ax.set_xlabel("Clock Efetivo Médio da Rodada (MHz)")
    ax.set_ylabel("Geekbench 6 Score Multi-Core")
    
    # Estilização de grades discretas para artigos científicos
    ax.grid(axis="both", linestyle=":", alpha=0.5, color="#aaaaaa")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    # Grava na subpasta correta de análise de dados detalhados (raw_plots)
    caminho_saida = caminhos.resultados_figuras / "raw_plots" / f"fig_dispersao_clock_score_{nome_maquina}.png"
    plt.savefig(caminho_saida, dpi=300)
    plt.close(fig)
    print(f"Gráfico de dispersão ({nome_maquina}) gerado com sucesso em: {caminho_saida}")


## Etapa 6 — Exportação de Tabelas para LaTeX

Geramos o código LaTeX das principais tabelas (formato `tabular`, sem
linhas verticais grossas, compatível com o padrão SBC) para colagem
direta no `main.tex` do Overleaf. **A legenda (`\caption`) deve ser
posicionada ANTES do `\begin{tabular}`**, conforme a diretriz do
professor — adicione-a manualmente ao colar o código gerado.


In [31]:
def gerar_latex_tabela(df: pd.DataFrame, legenda: str, rotulo: str) -> str:
    '''Gera um bloco de tabela LaTeX simples no padrão SBC (sem linhas verticais grossas).'''
    corpo_tabular = df.to_latex(index=False, escape=True, float_format="%.2f")
    bloco = (
        "\\begin{table}[ht]\n"
        "\\centering\n"
        f"\\caption{{{legenda}}}\n"
        f"\\label{{{rotulo}}}\n"
        f"{corpo_tabular}"
        "\\end{table}\n"
    )
    return bloco


if not tabela_comparativa_benchmark.empty:
    colunas_exportar = ["maquina", "Single_Core_media", "Single_Core_desvio_padrao",
                         "Multi_Core_media", "Multi_Core_desvio_padrao"]
    latex_benchmark = gerar_latex_tabela(
        tabela_comparativa_benchmark[colunas_exportar].rename(columns={
            "maquina": "Máquina",
            "Single_Core_media": "Single-Core (méd.)",
            "Single_Core_desvio_padrao": "Single-Core (dp)",
            "Multi_Core_media": "Multi-Core (méd.)",
            "Multi_Core_desvio_padrao": "Multi-Core (dp)",
        }),
        legenda="Resultados médios do Geekbench 6 por máquina. Fonte: Dados da pesquisa (2026).",
        rotulo="tab:benchmark_comparativo",
    )
    print(latex_benchmark)

    caminho_tex = caminhos.resultados_tabelas / "tabela_benchmark_comparativo.tex"
    caminho_tex.write_text(latex_benchmark, encoding="utf-8")
    print(f"\nCódigo LaTeX salvo em: {caminho_tex}")


\begin{table}[ht]
\centering
\caption{Resultados médios do Geekbench 6 por máquina. Fonte: Dados da pesquisa (2026).}
\label{tab:benchmark_comparativo}
\begin{tabular}{lrrrr}
\toprule
Máquina & Single-Core (méd.) & Single-Core (dp) & Multi-Core (méd.) & Multi-Core (dp) \\
\midrule
Máquina A & 2216.20 & 21.20 & 8116.50 & 135.26 \\
Máquina B & 1851.95 & 91.09 & 6107.70 & 155.71 \\
Máquina C & 932.00 & 95.94 & 2415.50 & 321.82 \\
Máquina D & 1322.45 & 9.63 & 2878.95 & 101.57 \\
Máquina F & 2828.30 & 21.82 & 14241.00 & 85.17 \\
\bottomrule
\end{tabular}
\end{table}


Código LaTeX salvo em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\tables\tabela_benchmark_comparativo.tex


In [32]:
if not tabela_eficiencia.empty:
    latex_eficiencia = gerar_latex_tabela(
        tabela_eficiencia.rename(columns={
            "maquina": "Máquina",
            "score_multicore_medio": "Score Multi-Core",
            "potencia_cpu_media_w": "Potência CPU (W)",
            "clock_efetivo_medio_mhz": "Clock efetivo (MHz)",
            "desempenho_por_watt": "Score/Watt",
            "ipc_relativo": "IPC relativo (proxy)",
            "pct_tempo_throttling": "% tempo throttling",
        }).drop(columns=["pasta"], errors="ignore"),
        legenda="Eficiência microarquitetural por máquina. Fonte: Dados da pesquisa (2026).",
        rotulo="tab:eficiencia_microarquitetural",
    )
    print(latex_eficiencia)

    caminho_tex = caminhos.resultados_tabelas / "tabela_eficiencia_microarquitetural.tex"
    caminho_tex.write_text(latex_eficiencia, encoding="utf-8")
    print(f"\nCódigo LaTeX salvo em: {caminho_tex}")


\begin{table}[ht]
\centering
\caption{Eficiência microarquitetural por máquina. Fonte: Dados da pesquisa (2026).}
\label{tab:eficiencia_microarquitetural}
\begin{tabular}{lrrrlll}
\toprule
Máquina & Score Multi-Core & Potência CPU (W) & Clock efetivo (MHz) & Score/Watt & IPC relativo (proxy) & \% tempo throttling \\
\midrule
Máquina A & 8116.50 & NaN & NaN & NaN & NaN & NaN \\
Máquina B & 6107.70 & NaN & NaN & NaN & NaN & NaN \\
Máquina C & 2415.50 & NaN & NaN & NaN & NaN & NaN \\
Máquina D & 2878.95 & NaN & NaN & NaN & NaN & NaN \\
\bottomrule
\end{tabular}
\end{table}


Código LaTeX salvo em: d:\Projetos\GitHub\UFPA\aoc-empirical-benchmark\results\tables\tabela_eficiencia_microarquitetural.tex


## Observações finais e próximos passos

- Os **alertas de completude** (Etapa 1) devem ser revisados antes da
  versão final do artigo: máquinas com menos de 20 rodadas tendem a ter
  estimativas de desvio padrão menos confiáveis.
- Um **desvio padrão (ou coeficiente de variação) elevado** no Score
  Multi-Core de uma máquina, associado a um **percentual alto de tempo em
  *throttling térmico*** (Etapa 3) e a quedas visíveis de clock efetivo
  (Etapa 5.7), é evidência arquitetural de que a variabilidade observada
  decorre de **regulagem térmica protetiva** do processador, não de ruído
  de medição — esse é o tipo de argumento que deve ser desenvolvido na
  Seção de Discussão do artigo, sustentado pelas figuras geradas aqui.
- Conforme novas pastas `machine_x` (ou arquivos dentro delas) forem
  adicionadas ao repositório, basta **reexecutar este notebook**: nenhuma
  alteração de código é necessária — o pipeline foi escrito para
  descobrir automaticamente todas as máquinas presentes em `data/raw/`.
- Antes da entrega final, revisar manualmente todas as legendas
  (`\caption`) no `.tex`, garantindo a citação **"Fonte: Os autores
  (2026)"** para os gráficos e tabelas gerados a partir dos dados
  coletados pelo próprio grupo.
